In [13]:
from models.ML.ParliamentaryVectorization import ParliamentaryVectorization
from models.ML.PULKMeans import PULKMeans

In [ ]:
subset = "dataset/parcanDeb-mp/150"

In [ ]:
# Asumiendo que tienes el código anterior en el mismo script o importado
# from vectorization_step import ParliamentaryVectorization (o copia la clase anterior aquí)

def run_pul_pipeline(subset_path):
    # 1. Vectorización Global
    pv = ParliamentaryVectorization(subset_path)
    pv.load_and_vectorize()
    
    # 2. Seleccionar un diputado de prueba (el que tenga más intervenciones para que sea interesante)
    # Buscamos el MP con más datos
    target_mp = max(pv.mp_indices, key=lambda k: len(pv.mp_indices[k]))
    print(f"\n--- Probando PUL-KM para el diputado: {target_mp} ---")
    
    # 3. Obtener matrices P y U
    P, U = pv.get_data_for_mp(target_mp)
    
    # 4. Ejecutar algoritmo del Paper
    pul_model = PULKMeans()
    rn_indices = pul_model.fit(P, U)
    
    # 5. Validación de Salida
    # Ahora tenemos:
    # - Positivos (P): Las intervenciones de target_mp
    # - Negativos Fiables (RN): Las filas de U indexadas por rn_indices
    RN_matrix = U[rn_indices]
    
    print(f"\nResumen Dataset Entrenamiento Final para {target_mp}:")
    print(f"Positivos (Clase 1): {P.shape[0]} documentos")
    print(f"Negativos (Clase 0): {RN_matrix.shape[0]} documentos (Negativos Fiables)")
    print(f"Descartados (Ruido): {U.shape[0] - RN_matrix.shape[0]} documentos (Ambiguos)")

In [ ]:
run_pul_pipeline(subset)

In [ ]:
# --- EJECUCIÓN ---
from models.ML.ParliamentaryClassifier import ParliamentaryClassifier

dataset = "dataset/parcanDeb-mp/"
all_subset = ["10", "25", "75", "150"]
for subset in all_subset:
    print(f"\n=== Ejecutando Pipeline Completo para Subconjunto: {subset} ===")
    pipeline = ParliamentaryClassifier(subset_path=dataset + subset)
    resultados = pipeline.run_full_pipeline()

In [ ]:
resultados

In [22]:

from models.ML.ParliamentaryClassifierGlobal import ParliamentaryClassifierGlobal

# quitar warnings por sklearn
import warnings
warnings.filterwarnings("ignore")


app = ParliamentaryClassifierGlobal("dataset/parcanDeb-mp/150")
app.run_evaluation()

1. Vectorizando y preparando datos...
--- Cargando datos de: dataset/parcanDeb-mp/150/train.json ---
-> Preparando corpus...
-> Entrenando TF-IDF en 30256 documentos...
   [OK] Matriz generada. Dimensiones: (30256, 35745)
   (Documentos: 30256, Vocabulario: 35745)
   -> Vectorizando 3836 documentos de test...

2. Entrenando 101 modelos SVM (uno por diputado)...
   [101/101] Procesando: Álvaro Lavanderaiesde Fuera De La Sede

3. Calculando métricas finales...

--- Evaluando 3836 documentos de Test ---

=== REPORTE DE EVALUACIÓN FINAL ===
>> CLASIFICACIÓN (Top-1 Prediction)
   Accuracy:            0.7508
   F1-Score (Weighted): 0.7507
   Precision (Weighted):0.7761
   Recall (Weighted):   0.7508
   (Macro F1: 0.7385)

>> RANKING / RECUPERACIÓN
   MRR:                 0.8269
   nDCG:                0.8671
   Recall@1 :           0.7508
   Recall@5 :           0.9241
   Recall@10:           0.9614


{'Accuracy': 0.7507820646506778,
 'Precision (Weighted)': np.float64(0.7761041221121663),
 'Recall (Weighted)': np.float64(0.7507820646506778),
 'F1-Score (Weighted)': np.float64(0.750699830026024),
 'Precision (Macro)': np.float64(0.7916635964949623),
 'Recall (Macro)': np.float64(0.7170526805230999),
 'F1-Score (Macro)': np.float64(0.7384577327823593),
 'MRR': np.float64(0.8269358235737353),
 'nDCG': np.float64(0.8671229518118241),
 'Recall@1': 0.7507820646506778,
 'Recall@5': 0.9241397288842544,
 'Recall@10': 0.9614181438998958}

In [3]:
from models.IR.IRClassifier import IRClassifier

ir_baseline = IRClassifier("dataset/parcanDeb-mp/all")
ir_baseline.run_evaluation()

--- INICIANDO BASELINE IR (Vector Space Model) ---
1. Vectorizando Corpus de Entrenamiento...
--- Cargando datos de: dataset/parcanDeb-mp/all/train.json ---
-> Preparando corpus...
-> Entrenando TF-IDF en 38242 documentos...
   [OK] Matriz generada. Dimensiones: (38242, 39965)
   (Documentos: 38242, Vocabulario: 39965)
2. Construyendo perfiles vectoriales (Centroides)...
   [OK] Perfiles creados para 454 diputados.
3. Vectorizando 4945 documentos de Test...
4. Calculando Similitud Coseno (Query vs Profiles)...

5. Calculando métricas...

--- Evaluando 4945 documentos de Test ---

=== REPORTE DE EVALUACIÓN FINAL (COMPARABLE CON PAPER) ===
>> MÉTRICAS GLOBALES (MICRO)
   Accuracy:            0.6032
   Micro Precision:     0.6032
   Micro Recall:        0.6032
   Micro F1-Score:      0.6032  <-- DATO CLAVE PAPER

>> MÉTRICAS PROMEDIO (MACRO)
   Macro Precision:     0.4298
   Macro Recall:        0.4070
   Macro F1-Score:      0.3961  <-- DATO CLAVE PAPER

>> RANKING / IR
   MRR:          

{'Accuracy': 0.6032355915065722,
 'Precision (Micro)': np.float64(0.6032355915065722),
 'Recall (Micro)': np.float64(0.6032355915065722),
 'F1-Score (Micro)': np.float64(0.6032355915065722),
 'Precision (Macro)': np.float64(0.4298323599672964),
 'Recall (Macro)': np.float64(0.4069653203643346),
 'F1-Score (Macro)': np.float64(0.39605724544491977),
 'Precision (Weighted)': np.float64(0.6491792016788374),
 'Recall (Weighted)': np.float64(0.6032355915065722),
 'F1-Score (Weighted)': np.float64(0.6050313366118257),
 'MRR': np.float64(0.6990403626727613),
 'nDCG': np.float64(0.7632289501294366),
 'Recall@1': 0.6032355915065722,
 'Recall@5': 0.8161779575328615,
 'Recall@10': 0.8810920121334681}